In [0]:
-- revenue_by_date
-- Must return ZERO rows.
(
  SELECT 'missing_from_gold' AS problem, pickup_date, total_revenue, trip_count
  FROM nyc_mobility.gold_expected.revenue_by_date
  EXCEPT
  SELECT 'missing_from_gold' AS problem, pickup_date, total_revenue, trip_count
  FROM nyc_mobility.gold.revenue_by_date
)
UNION ALL
(
  SELECT 'unexpected_in_gold' AS problem, pickup_date, total_revenue, trip_count
  FROM nyc_mobility.gold.revenue_by_date
  EXCEPT
  SELECT 'unexpected_in_gold' AS problem, pickup_date, total_revenue, trip_count
  FROM nyc_mobility.gold_expected.revenue_by_date
);



-- 2. Conservation checks. Expected: 2,723,707 four times, and 74657443.24 three times.

-- Cross-validation: every Gold table accounts for every Silver trip
SELECT
  (SELECT COUNT(*)        FROM nyc_mobility.silver.trips)               AS silver_rows,
  (SELECT SUM(trip_count) FROM nyc_mobility.gold.revenue_by_date)        AS via_dates,
  (SELECT SUM(trip_count) FROM nyc_mobility.gold.trips_by_borough)       AS via_boroughs,
  (SELECT SUM(trip_count) FROM nyc_mobility.gold.trips_by_hour_weekend)  AS via_hours;

-- Cross-validation: total dollar revenue conserved across grains
SELECT
  (SELECT ROUND(SUM(CAST(total_amount AS DECIMAL(12,2))), 2)
     FROM nyc_mobility.silver.trips)                        AS silver_revenue,
  (SELECT ROUND(SUM(total_revenue), 2)
     FROM nyc_mobility.gold.revenue_by_date)                AS via_dates,
  (SELECT ROUND(SUM(total_revenue), 2)
     FROM nyc_mobility.gold.trips_by_borough)               AS via_boroughs;
-- 3. The three Gold anchors (added in Lecture 74)

-- Anchor 1: total revenue, from the Gold table
SELECT
  ROUND(SUM(total_revenue), 2) AS total_january_revenue,
  SUM(trip_count)              AS total_trips,
  COUNT(*)                     AS days_in_dataset
FROM nyc_mobility.gold.revenue_by_date;

-- Anchor 2: borough ranking with share of trips
SELECT
  pickup_borough,
  trip_count,
  ROUND(100.0 * trip_count / SUM(trip_count) OVER (), 2) AS pct_of_trips
FROM nyc_mobility.gold.trips_by_borough
ORDER BY trip_count DESC;

-- Anchor 3: peak hours ranked within each day type
SELECT
  is_weekend,
  pickup_hour,
  trip_count,
  RANK() OVER (
    PARTITION BY is_weekend
    ORDER BY trip_count DESC
  ) AS hour_rank_within_day_type
FROM nyc_mobility.gold.trips_by_hour_weekend
QUALIFY hour_rank_within_day_type <= 4
ORDER BY is_weekend DESC, hour_rank_within_day_type;
